# Retrieval Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
import pickle
cleaned_docs = []
with open("cleaned_docs.pkl", "rb") as f:
    cleaned_docs = pickle.load(f)

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=hf_embeddings, 
)

In [5]:
TEST_QUESTIONS = [
    {
        "id": "Q01", "level": 1, "source": "WHO",
        "question": "What are the three classes of pharmacological antihypertensive medications recommended by the WHO as an initial treatment?",
        "aspect_tested": "Direct list retrieval",
        "expected_answer": "Thiazide and thiazide-like agents, ACE inhibitors (ACEis)/angiotensin-receptor blockers (ARBs), and long-acting dihydropyridine calcium channel blockers (CCBs)."
    },
    {
        "id": "Q02", "level": 1, "source": "NICE",
        "question": 'According to the NICE guideline, what is the exact definition of "accelerated hypertension"?',
        "aspect_tested": "Glossary definition / exact terminology",
        "expected_answer": "A severe increase in blood pressure to 180/120 mmHg or higher with signs of retinal haemorrhage and/or papilloedema."
    },
    {
        "id": "Q03", "level": 1, "source": "NICE",
        "question": "What specific life-threatening symptoms require a same-day specialist referral under the NICE guidelines for a patient with a clinic blood pressure of 180/120 mmHg or higher?",
        "aspect_tested": "Specific criteria identification",
        "expected_answer": "New onset confusion, chest pain, signs of heart failure, or acute kidney injury."
    },
    {
        "id": "Q04", "level": 1, "source": "WHO",
        "question": "What four conditions must be met for nonphysician professionals to provide pharmacological treatment for hypertension according to the WHO?",
        "aspect_tested": "Strict preconditions within a specific section",
        "expected_answer": "Proper training, prescribing authority, specific management protocols, and physician oversight."
    },
    {
        "id": "Q05", "level": 1, "source": "NICE",
        "question": 'How does the NICE guideline define the "white-coat effect"?',
        "aspect_tested": "Concept definition",
        "expected_answer": "A discrepancy of more than 20/10 mmHg between clinic and average daytime ABPM or average HBPM blood pressure measurements at the time of diagnosis."
    },
    {
        "id": "Q06", "level": 2, "source": "NICE",
        "question": "If a patient's clinic blood pressure is 140/90 mmHg or higher, what are the exact sequential steps for taking further measurements during the consultation according to NICE?",
        "aspect_tested": "Procedural logic / sequential steps",
        "expected_answer": "Take a second measurement during the consultation. If the second measurement is substantially different from the first, take a third measurement. Record the lower of the last 2 measurements as the clinic blood pressure."
    },
    {
        "id": "Q07", "level": 2, "source": "WHO",
        "question": "Does the WHO recommend delaying the start of pharmacological treatment in order to perform a formal cardiovascular disease (CVD) risk assessment?",
        "aspect_tested": "Negative constraint / contextual caveats",
        "expected_answer": "No — CVD risk assessment is suggested only where feasible and should not delay treatment. If it may threaten timely initiation, it should be postponed and included in the follow-up strategy."
    },
    {
        "id": "Q08", "level": 2, "source": "NICE",
        "question": "What specific warnings does the NICE guideline provide regarding the use of salt substitutes containing potassium chloride?",
        "aspect_tested": "Exceptions and edge cases",
        "expected_answer": "Should not be used by older people, people with diabetes, pregnant women, people with kidney disease, and people taking some antihypertensive drugs such as ACE inhibitors and angiotensin II receptor blockers."
    },
    {
        "id": "Q09", "level": 2, "source": "WHO",
        "question": "What is the WHO's target systolic blood pressure treatment goal for high-risk patients (those with high CVD risk, diabetes mellitus, or chronic kidney disease)?",
        "aspect_tested": "Conditional numerical thresholds",
        "expected_answer": "The target systolic blood pressure goal is <130 mmHg."
    },
    {
        "id": "Q10", "level": 2, "source": "NICE",
        "question": "How does the NICE guideline differentiate the clinic blood pressure targets for adults aged under 80 compared to those aged 80 and over?",
        "aspect_tested": "Age-based threshold comparison",
        "expected_answer": "Under 80: target below 140/90 mmHg. 80 and over: target below 150/90 mmHg."
    },
    {
        "id": "Q11", "level": 3, "source": "NICE",
        "question": "Under the NICE Step 1 treatment recommendations, what is the correct initial medication for a 60-year-old patient of Black African family origin who does not have type 2 diabetes?",
        "aspect_tested": "Multi-variable conditional logic (age + ethnicity + comorbidity)",
        "expected_answer": "A calcium-channel blocker (CCB) should be offered."
    },
    {
        "id": "Q12", "level": 3, "source": "WHO",
        "question": 'According to the WHO\'s "Algorithm 2" (initiation of treatment not using a single-pill combination), what is the next step if a patient starting on a half-maximal dose of a CCB (like Amlodipine 5 mg) is not at their blood pressure goal after 4-6 weeks?',
        "aspect_tested": "Following algorithmic flowcharts / decision trees",
        "expected_answer": "Increase the CCB by doubling the dose (e.g., to Amlodipine 10 mg once a day)."
    },
    {
        "id": "Q13", "level": 3, "source": "NICE",
        "question": "In the NICE guideline's Step 4 treatment for resistant hypertension, how does the patient's blood potassium level dictate the choice of the fourth antihypertensive drug?",
        "aspect_tested": "Lab-value-dependent treatment branching",
        "expected_answer": "If potassium ≤4.5 mmol/l: consider further diuretic therapy with low-dose spironolactone. If potassium >4.5 mmol/l: consider an alpha-blocker or beta-blocker."
    },
    {
        "id": "Q14", "level": 3, "source": "WHO",
        "question": "What are the maternal and fetal risks of using renin-angiotensin-aldosterone system inhibitors (like ACEis or ARBs) during pregnancy according to the WHO, and how does this affect treatment algorithms?",
        "aspect_tested": "Synthesizing pathophysiology warnings with treatment contraindications",
        "expected_answer": "Associated with serious fetal toxicity, including renal and cardiac abnormalities and death. Strictly contraindicated in pregnancy — neither ACEi nor ARB should be given to pregnant women."
    },
    {
        "id": "Q15", "level": 3, "source": "NICE",
        "question": "How should a healthcare professional appropriately measure and manage blood pressure for a patient presenting with symptoms of postural hypotension, according to the NICE guideline?",
        "aspect_tested": "Multi-step clinical workflow and differential response logic",
        "expected_answer": "Measure lying/seated, then again after standing ≥1 minute. If systolic drops ≥20 mmHg or diastolic ≥10 mmHg: review medication, manage falls risk, measure subsequent readings standing, and consider specialist referral if symptoms persist."
    },
]

import pandas as pd
df_questions = pd.DataFrame(TEST_QUESTIONS)
df_questions.head()

,id,level,source,question,aspect_tested,expected_answer
0,Q01,1,WHO,What are the three classes of pharmacological ...,Direct list retrieval,"Thiazide and thiazide-like agents, ACE inhibit..."
1,Q02,1,NICE,"According to the NICE guideline, what is the e...",Glossary definition / exact terminology,A severe increase in blood pressure to 180/120...
2,Q03,1,NICE,What specific life-threatening symptoms requir...,Specific criteria identification,"New onset confusion, chest pain, signs of hear..."
3,Q04,1,WHO,What four conditions must be met for nonphysic...,Strict preconditions within a specific section,"Proper training, prescribing authority, specif..."
4,Q05,1,NICE,"How does the NICE guideline define the ""white-...",Concept definition,A discrepancy of more than 20/10 mmHg between ...


In [6]:
def retrieve_topk(query, k, vs=vectorstore):
    return vs.similarity_search_with_score(query, k=k)

In [7]:
def show_topk_comparison(question, ks=(3, 5, 10)):
    print(f"\n{'='*90}\nQUERY: {question}\n{'='*90}")
    for k in ks:
        results = retrieve_topk(question, k)
        print(f"\n--- Top-{k} ---")
        for rank, (doc, score) in enumerate(results, 1):
            meta = doc.metadata
            preview = doc.page_content[:400].replace("\n", " ")
            print(f"[{rank}] score={score:.4f} | "
                  f"{meta.get('document_name', '?')} "
                  f"page:{meta.get('page_number', '?')} "
                  f"(chunk_id={meta.get('chunk_id', '?')})")
            print(f"     {preview}...")

In [8]:
for qid in ["Q01", "Q06", "Q11"]:
    q = next(item["question"] for item in TEST_QUESTIONS if item["id"] == qid)
    show_topk_comparison(q, ks=(3, 5, 10))


QUERY: What are the three classes of pharmacological antihypertensive medications recommended by the WHO as an initial treatment?

--- Top-3 ---
[1] score=0.4778 | file1.pdf page:10 (chunk_id=file1.pdf_ch0003)
     WHO suggests pharmacological antihypertensive treatment of individuals without  cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney  disease, and systolic blood pressure of 130–139 mmHg. Conditional recommendation, moderate- to high-certainty evidence  2. RECOMMENDATION ON LABORATORY TESTING When starting pharmacological therapy for hypertension, WHO sugg...
[2] score=0.4800 | file1.pdf page:9 (chunk_id=file1.pdf_ch0001)
     Executive summary More people die each year from cardiovascular diseases than from any other cause. Over three  quarters of heart disease and stroke-related deaths occur in low-income and middle-income countries.  Hypertension – or elevated blood pressure – is a serious medical condition that significantly inc

In [31]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from utils import prepare_metadata

CONFIG_A = {"chunk_size": 600, "chunk_overlap": 100, "name": "A_600_100"}
CONFIG_B = {"chunk_size": 800, "chunk_overlap": 200, "name": "B_800_200"}
CONFIG_C = {"chunk_size": 900, "chunk_overlap": 150, "name": "C_900_150"}
CONFIG_D = {"chunk_size": 400, "chunk_overlap": 50, "name": "D_400_50"}

splitter_a = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CONFIG_A["chunk_size"],
    chunk_overlap=CONFIG_A["chunk_overlap"],
)

splitter_b = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CONFIG_B["chunk_size"],
    chunk_overlap=CONFIG_B["chunk_overlap"],
)

splitter_c = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CONFIG_C["chunk_size"],
    chunk_overlap=CONFIG_C["chunk_overlap"],
)

splitter_d = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CONFIG_D["chunk_size"],
    chunk_overlap=CONFIG_D["chunk_overlap"],
)

chunks_a = splitter_a.split_documents(cleaned_docs)
chunks_b = splitter_b.split_documents(cleaned_docs)
chunks_c = splitter_c.split_documents(cleaned_docs)
chunks_d = splitter_d.split_documents(cleaned_docs)

chunks_a = prepare_metadata(chunks_a)
chunks_b = prepare_metadata(chunks_b)
chunks_c = prepare_metadata(chunks_c)
chunks_d = prepare_metadata(chunks_d)

print(f"Config A chunk count: {len(chunks_a)}")
print(f"Config B chunk count: {len(chunks_b)}")
print(f"Config C chunk count: {len(chunks_c)}")
print(f"Config D chunk count: {len(chunks_d)}")

Config A chunk count: 140
Config B chunk count: 121
Config C chunk count: 112
Config D chunk count: 232


In [10]:
vectorstore_a = Chroma.from_documents(
    chunks_a,
    hf_embeddings,
    persist_directory="chroma_db_configA",
    collection_name="config_a",
    collection_metadata={"hnsw:space": "cosine"},
)

# vectorstore_a = Chroma(
#     persist_directory="./chroma_db_configA",
#     embedding_function=hf_embeddings,
#     collection_name="config_a",
#      collection_metadata={"hnsw:space": "cosine"}, 
# )

In [11]:
vectorstore_b = Chroma.from_documents(
    chunks_b,
    hf_embeddings,
    persist_directory="chroma_db_configB",
    collection_name="config_b",
    collection_metadata={"hnsw:space": "cosine"}
)
# vectorstore_b = Chroma(
#     persist_directory="./chroma_db_configB",
#     embedding_function=hf_embeddings,
#     collection_name="config_b" 
# )

In [12]:
vectorstore_c = Chroma.from_documents(
    chunks_c,
    hf_embeddings,
    persist_directory="chroma_db_configC",
    collection_name="config_c",
    collection_metadata={"hnsw:space": "cosine"}
)

# vectorstore_c = Chroma(
#     persist_directory="./chroma_db_configC",
#     embedding_function=hf_embeddings,
#     collection_name="config_c" 
# )

In [32]:
vectorstore_d = Chroma.from_documents(
    chunks_d,
    hf_embeddings,
    persist_directory="chroma_db_configD",
    collection_name="config_d",
    collection_metadata={"hnsw:space": "cosine"}
)

In [33]:
vectorstore_d.similarity_search("what's hypertension?")

[Document(id='dc0d72c3-b3ea-4bf2-9feb-14d19bd3368f', metadata={'page_number': 13, 'document_name': 'file1.pdf', 'chunk_id': 'file1.pdf_ch0006', 'source_url': './files\\file1.pdf'}, page_content='1 Introduction\nMore people die each year from cardiovascular diseases than from any other cause. Over three quarters \nof heart disease and stroke-related deaths occur in low-income and middle-income countries (1). Blood \npressure is the force exerted by circulating blood against the walls of the body’s arteries, the major blood \nvessels in the body. Blood pressure is written as two numbers. The first number (systolic) represents the \npressure in blood vessels when the heart contracts or beats. The second number (diastolic) represents \nthe pressure in the vessels when the heart rests between beats. Hypertension – or elevated blood \npressure – is a serious medical condition that significantly increases the risk of diseases of the heart, \nbrain, kidneys and other organs (2). Hypertension c

In [14]:
def compare_configs(question, k=5):
    print(f"\n{'='*90}\nQUERY: {question}\n{'='*90}")
    for name, vs in [(CONFIG_A["name"], vectorstore_a), (CONFIG_B["name"], vectorstore_b)]:
        print(f"\n--- Config {name} (Top-{k}) ---")
        for rank, (doc, score) in enumerate(retrieve_topk(question, k, vs), 1):
            m = doc.metadata
            print(f"[{rank}] score={score:.4f} | file: {m.get('document_name')} | page: {m.get('page_number')} | " 
                  f"chunk_id={m.get('chunk_id')}")
            print(f"     {doc.page_content.replace(chr(10), ' ')}...")

for qid in ["Q01", "Q06", "Q11"]:
    q = next(x["question"] for x in TEST_QUESTIONS if x["id"] == qid)
    compare_configs(q, k=5)


QUERY: What are the three classes of pharmacological antihypertensive medications recommended by the WHO as an initial treatment?

--- Config A_600_100 (Top-5) ---
[1] score=0.2389 | file: file1.pdf | page: 10 | chunk_id=file1.pdf_ch0003
     WHO suggests pharmacological antihypertensive treatment of individuals without  cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney  disease, and systolic blood pressure of 130–139 mmHg. Conditional recommendation, moderate- to high-certainty evidence  2. RECOMMENDATION ON LABORATORY TESTING When starting pharmacological therapy for hypertension, WHO suggests obtaining tests to  screen for comorbidities and secondary hypertension, but only when testing does not delay  or impede starting treatment. Conditional recommendation, low-certainty evidence 3. RECOMMENDATION ON CARDIOVASCULAR DISEASE RISK ASSESSMENT  WHO suggests cardiovascular disease risk assessment at or after the initiation of  pharmacological

# Relevance Labels (Top-5)

## Config A_600_100

| Question | Rank1 | Rank2 | Rank3 | Rank4 | Rank5 |
|---|---|---|---|---|---|
| Q01 | 1 | 0 | 1 | 0 | 0 |
| Q06 | 1 | 0 | 0 | 0 | 0 |
| Q11 | 0 | 0 | 0 | 0 | 0 |

## Config B_800_200

| Question | Rank1 | Rank2 | Rank3 | Rank4 | Rank5 |
|---|---|---|---|---|---|
| Q01 | 1 | 1 | 0 | 1 | 0 |
| Q06 | 1 | 0 | 0 | 0 | 0 |
| Q11 | 0 | 0 | 0 | 0 | 0 |

## Percision@5 VS. Percision@3

#### Config A_600_100

| Question | Precision@3 | Precision@5 |
|---|---|---|
| Q01 | 0.67 | 0.40 |
| Q06 | 0.33 | 0.20 |
| Q11 | 0.00 | 0.00 |
| **Average** | **0.33** | **0.20** |

#### Config B_800_200

| Question | Precision@3 | Precision@5 |
|---|---|---|
| Q01 | 0.67 | 0.60 |
| Q06 | 0.33 | 0.20 |
| Q11 | 0.00 | 0.00 |
| **Average** | **0.33** | **0.27** |

In [15]:
vectorstore_b.similarity_search_with_score("Under the NICE Step 1 treatment recommendations, what is the correct initial medication for a 60-year-old patient of Black African family origin who does not have type 2 diabetes?", k=15)

[(Document(id='4e436ee7-faa0-411b-87bc-4794261b47be', metadata={'chunk_id': 'file2.pdf_ch0112', 'source_url': './files\\file2.pdf', 'document_name': 'file2.pdf', 'page_number': 43}, page_content='people of Black African or African–Caribbean family origin). The committee discussed the \nevidence for this and agreed that it was sufficient to support and retain this \nrecommendation. The committee agreed it should be broadened to include the choice of \nan ACE inhibitor or an angiotensin\xa0II receptor blocker (ARB; also referred to as A-type \ndrugs), because they are now cost equivalent, and the committee also agreed they are \nclinically equivalent. \nFor people of Black African or African–Caribbean family origin with type\xa02 diabetes, the \nprevious recommendation was to offer step\xa01 dual therapy with an ACE inhibitor and either \na diuretic (D-type drug) or a calcium channel blocker (CCB; C-type drug). However, these \nrecommendations were based on monotherapy studies and when t

### Retrieval Failure Documentation: Q11

| Evaluation Field | Details |
| :--- | :--- |
| **Question (Q11)** | *Under the NICE Step 1 treatment recommendations, what is the correct initial medication for a 60-year-old patient of Black African family origin who does not have type 2 diabetes?* |
| **Failure Mode** | **Correct chunk ranked too low** |
| **Symptom** | The right chunk exists in the database and was retrieved, but it sits at rank 6, missing the top 5 cutoff to be sent to the LLM. The top 5 slots were filled with irrelevant, high-similarity chunks. |
| **Expected Content** | The actionable bullet point on page 20 of `file2.pdf` (chunk `file2.pdf_ch0089`) stating: "Offer a calcium-channel blocker (CCB) to adults starting step 1 antihypertensive treatment who... are aged 55 or over... or are of Black African or African–Caribbean family origin and do not have type 2 diabetes." |
| **Root Cause Analysis** | The semantic search heavily weighted dense narrative paragraphs from the "Rationale and impact" sections (pages 42-43) that repeated the query's keywords ("Step 1 treatment", "type 2 diabetes", "Black African"). This semantic overlap pushed the concise, actionable clinical rule down to position 6. |
| **Mitigation Strategy / Fix** | Apply **reranking** to evaluate the logical context of the top 10-15 results, implement **better chunking** (e.g., separating rationale sections from actionable recommendations), or upgrade to a **stronger embedding model**. |

## Implement reranking

In [16]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [17]:
def rerank(query, k_final=5, pool_size=15, vs=vectorstore):
    
    candidates = retrieve_topk(query, k=pool_size, vs=vs)
    
    pairs = [[query, doc.page_content] for doc, _ in candidates]
    scores = cross_encoder.predict(pairs)
    
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(doc, rr_score) for ((doc, _), rr_score) in reranked[:k_final]]



In [18]:
q11 = next(x["question"] for x in TEST_QUESTIONS if x["id"] == "Q11")
reranked_results = rerank(q11, k_final=5, pool_size=15, vs=vectorstore_b)

for rank, (doc, score) in enumerate(reranked_results, 1):
    m = doc.metadata
    print(f"[{rank}] rerank_score={score:.4f} p.{m.get('page_number')} chunk_id={m.get('chunk_id')}")
    print(f"     {doc.page_content.replace(chr(10), ' ')}...")

[1] rerank_score=2.9699 p.43 chunk_id=file2.pdf_ch0112
     people of Black African or African–Caribbean family origin). The committee discussed the  evidence for this and agreed that it was sufficient to support and retain this  recommendation. The committee agreed it should be broadened to include the choice of  an ACE inhibitor or an angiotensin II receptor blocker (ARB; also referred to as A-type  drugs), because they are now cost equivalent, and the committee also agreed they are  clinically equivalent.  For people of Black African or African–Caribbean family origin with type 2 diabetes, the  previous recommendation was to offer step 1 dual therapy with an ACE inhibitor and either  a diuretic (D-type drug) or a calcium channel blocker (CCB; C-type drug). However, these  recommendations were based on monotherapy studies and when the committee looked at  this evidence alongside the new dual therapy evidence review, they concluded that it was  insufficient to recommend starting dual 

## The Result
The Initial Search (The Problem): Your standard vector search was easily tricked by keyword density. The background "Rationale" sections repeated your query's keywords ("Black African," "type 2 diabetes," "Step 1") so often that the system thought they were the best match. This keyword overload pushed the actual, concise clinical rule down to rank 6.

The Reranking (The Fix): The reranker stepped in like a reading comprehension judge. Instead of just matching concepts, it evaluated the logical relationship between your exact question and the text. It recognized that the chunk on page 20 actually provided the actionable answer to your specific conditions (especially the "does not have type 2 diabetes" part), successfully bumping it up into your top 3 results.

In [19]:
retrieve_topk(TEST_QUESTIONS[1]["question"], 17, vs=vectorstore_b)

[(Document(id='a5d70c2a-bf9d-40ec-b859-53b173e5ff04', metadata={'chunk_id': 'file2.pdf_ch0094', 'source_url': './files\\file2.pdf', 'page_number': 25, 'document_name': 'file2.pdf'}, page_content='Terms used in this guideline \nThis section defines terms that have been used in a particular way for this guideline. For \nother definitions see the NICE glossary. \nAccelerated hypertension \nA severe increase in blood pressure to 180/120\xa0mmHg or higher (and often over 220/\n120\xa0mmHg) with signs of retinal haemorrhage and/or papilloedema (swelling of the optic \nnerve). It is usually associated with new or progressive target organ damage and is also \nknown as malignant hypertension. \nEstablished cardiovascular disease \nMedical history of ischaemic heart disease, cerebrovascular disease, peripheral vascular \ndisease, aortic aneurysm or heart failure. Cardiovascular disease is a general term for \nconditions affecting the heart or blood vessels. It is usually associated with a build-

In [20]:
from typing import List, Dict

def crossencoder_label(
    expected_answer: str,
    chunk: str,
    threshold: float = 1.0
) -> bool:

    score = cross_encoder.predict(
        [(expected_answer, chunk)]
    )

    return float(score[0]) >= threshold


def label_all_chunks(
    expected_answer: str,
    chunks: List[str],
    threshold: float = 1.0
) -> List[Dict]:

    if not expected_answer or not expected_answer.strip():
        raise ValueError("expected_answer cannot be empty.")

    if not chunks:
        return []

    valid_chunks = [
        chunk for chunk in chunks
        if chunk and chunk.strip()
    ]

    if not valid_chunks:
        return []


    pairs = [
        (expected_answer, chunk)
        for chunk in valid_chunks
    ]

    scores = cross_encoder.predict(pairs)

    results = []

    for chunk, score in zip(valid_chunks, scores):
        score = float(score)

        results.append({
            "chunk": chunk,
            "score": score,
            "relevant": score >= threshold
        })

    return results





expected_answer = TEST_QUESTIONS[1]["expected_answer"]
chunks = [doc.page_content for doc in vectorstore_b.similarity_search(TEST_QUESTIONS[1]["question"],k=30)]

results = label_all_chunks(
    expected_answer=expected_answer,
    chunks=chunks,
    threshold=1.0
)


# Display results
for i, result in enumerate(results, start=1):
    print(f"\n{'=' * 70}")
    print(f"Chunk {i}")
    print(f"Score:     {result['score']:.4f}")
    print(f"Relevant:  {result['relevant']}")
    # print(f"Text:      {result['chunk']}")


Chunk 1
Score:     8.0988
Relevant:  True

Chunk 2
Score:     -9.9059
Relevant:  False

Chunk 3
Score:     -8.2277
Relevant:  False

Chunk 4
Score:     -8.5113
Relevant:  False

Chunk 5
Score:     -7.8747
Relevant:  False

Chunk 6
Score:     -7.9704
Relevant:  False

Chunk 7
Score:     -11.0252
Relevant:  False

Chunk 8
Score:     -7.0420
Relevant:  False

Chunk 9
Score:     -1.2765
Relevant:  False

Chunk 10
Score:     -9.7018
Relevant:  False

Chunk 11
Score:     -5.1790
Relevant:  False

Chunk 12
Score:     -7.8863
Relevant:  False

Chunk 13
Score:     -9.1765
Relevant:  False

Chunk 14
Score:     -9.9225
Relevant:  False

Chunk 15
Score:     -10.0890
Relevant:  False

Chunk 16
Score:     -10.8835
Relevant:  False

Chunk 17
Score:     2.8196
Relevant:  True

Chunk 18
Score:     -6.5067
Relevant:  False

Chunk 19
Score:     -2.6853
Relevant:  False

Chunk 20
Score:     -8.7898
Relevant:  False

Chunk 21
Score:     -9.2766
Relevant:  False

Chunk 22
Score:     -5.5729
Relevant:  Fals

In [21]:
from typing import List, Dict

def crossencoder_label(
    expected_answer: str,
    chunk: str,
    threshold: float = 1.0
) -> bool:
    """Return True when the chunk is relevant to the expected answer."""
    score = cross_encoder.predict([(expected_answer, chunk)])
    return float(score[0]) >= threshold


def label_all_chunks(
    expected_answer: str,
    chunks: List[str],
    threshold: float = 1.0
) -> List[Dict]:
    """
    Evaluate and label all retrieved chunks.
    Returns a list containing the chunk, Cross-Encoder score,
    and relevance label.
    """
    if not expected_answer or not expected_answer.strip():
        raise ValueError("expected_answer cannot be empty.")

    if not chunks:
        return []

    valid_chunks = [chunk for chunk in chunks if chunk and chunk.strip()]

    if not valid_chunks:
        return []

    pairs = [(expected_answer, chunk) for chunk in valid_chunks]
    scores = cross_encoder.predict(pairs)

    results = []
    for chunk, score in zip(valid_chunks, scores):
        score = float(score)
        results.append({
            "chunk": chunk,
            "score": score,
            "relevant": score >= threshold,
        })

    return results



In [22]:
def precision_at_k(relevant_labels: List[bool], k: int) -> float:
    """Compute Precision@K from a list of boolean relevance labels."""
    top_k = relevant_labels[:k]
    if len(top_k) == 0:
        return 0.0
    return sum(top_k) / len(top_k)

def recall_at_k(relevant_in_topk: int, total_relevant: int) -> float:
    if total_relevant == 0:
        return 0.0
    return min(relevant_in_topk, total_relevant) / total_relevant

def f1_at_k(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def count_total_relevant(expected_answer: str, vectorstore, threshold: float = 1.0) -> int:
    all_docs = vectorstore.get()
    all_chunks = all_docs["documents"]
    labeled = label_all_chunks(expected_answer, all_chunks, threshold)
    return sum(1 for r in labeled if r["relevant"])

In [34]:
from itertools import product

K_VALUES = [3, 5, 7]
CONFIG_NAMES = ["A_600_100", "B_800_200", "C_900_150", "D_400_50"]
THRESHOLD = 1.0  # Cross-Encoder relevance threshold

VECTORSTORES = {
    "A_600_100": vectorstore_a,
    "B_800_200": vectorstore_b,
    "C_900_150": vectorstore_c,
    "D_400_50": vectorstore_d
}

print(f"Grid Search Configuration:")
print(f"  k values:       {K_VALUES}")
print(f"  Configs:        {CONFIG_NAMES}")
print(f"  Questions:      {len(TEST_QUESTIONS)}")
print(f"  Threshold:      {THRESHOLD}")
print(f"  Combinations:   {len(K_VALUES) * len(CONFIG_NAMES)}")


Grid Search Configuration:
  k values:       [3, 5, 7]
  Configs:        ['A_600_100', 'B_800_200', 'C_900_150', 'D_400_50']
  Questions:      15
  Threshold:      1.0
  Combinations:   12


In [35]:
# Precompute total relevant counts once per (config, question)
TOTAL_RELEVANT_CACHE = {}

for config_name in CONFIG_NAMES:
    vs = VECTORSTORES[config_name]
    TOTAL_RELEVANT_CACHE[config_name] = {}

    print(f"\nPrecomputing total relevant for config: {config_name}")
    for q in TEST_QUESTIONS:
        qid = q["id"]
        expected_answer = q["expected_answer"]

        total_rel = count_total_relevant(expected_answer, vs, THRESHOLD)
        TOTAL_RELEVANT_CACHE[config_name][qid] = total_rel

        print(f"  {qid}: total_relevant={total_rel}")

print("\nTotal relevant cache ready.")


Precomputing total relevant for config: A_600_100
  Q01: total_relevant=12
  Q02: total_relevant=3
  Q03: total_relevant=1
  Q04: total_relevant=2
  Q05: total_relevant=3
  Q06: total_relevant=1
  Q07: total_relevant=4
  Q08: total_relevant=2
  Q09: total_relevant=9
  Q10: total_relevant=7
  Q11: total_relevant=2
  Q12: total_relevant=3
  Q13: total_relevant=1
  Q14: total_relevant=4
  Q15: total_relevant=1

Precomputing total relevant for config: B_800_200
  Q01: total_relevant=10
  Q02: total_relevant=3
  Q03: total_relevant=1
  Q04: total_relevant=1
  Q05: total_relevant=3
  Q06: total_relevant=1
  Q07: total_relevant=5
  Q08: total_relevant=2
  Q09: total_relevant=8
  Q10: total_relevant=7
  Q11: total_relevant=5
  Q12: total_relevant=2
  Q13: total_relevant=1
  Q14: total_relevant=3
  Q15: total_relevant=1

Precomputing total relevant for config: C_900_150
  Q01: total_relevant=8
  Q02: total_relevant=3
  Q03: total_relevant=1
  Q04: total_relevant=2
  Q05: total_relevant=3
  Q06

In [36]:
import pandas as pd

all_rows = []
grid_results = []

for config_name, k in product(CONFIG_NAMES, K_VALUES):
    vs = VECTORSTORES[config_name]
    precisions = []
    recalls = []
    f1s = []

    print(f"\n{'─' * 70}")
    print(f"Config: {config_name} | k={k}")
    print(f"{'─' * 70}")

    for q in TEST_QUESTIONS:
        qid = q["id"]
        question = q["question"]
        expected_answer = q["expected_answer"]

        # docs = vs.similarity_search(question, k=k)
        docs = [doc[0] for doc in rerank(question, k_final=k, pool_size=40, vs=vs)]
        chunks = [doc.page_content for doc in docs]

        labeled = label_all_chunks(
            expected_answer=expected_answer,
            chunks=chunks,
            threshold=THRESHOLD,
        )

        relevance_labels = [r["relevant"] for r in labeled]

        p_at_k = precision_at_k(relevance_labels, k)

        total_rel = TOTAL_RELEVANT_CACHE[config_name][qid]
        r_at_k = recall_at_k(sum(relevance_labels), total_rel)
        f1_k = f1_at_k(p_at_k, r_at_k)

        precisions.append(p_at_k)
        recalls.append(r_at_k)
        f1s.append(f1_k)

        chunk_scores = [r["score"] for r in labeled]
        chunk_relevance = [r["relevant"] for r in labeled]

        print(f"  {qid}: P@{k}={p_at_k:.4f}  R@{k}={r_at_k:.4f}  F1@{k}={f1_k:.4f}  "
              f"[relevant: {sum(chunk_relevance)}/{len(chunk_relevance)}, total_rel: {total_rel}]")

        all_rows.append({
            "config": config_name,
            "k": k,
            "question_id": qid,
            "question_level": q["level"],
            "question_source": q["source"],
            "precision_at_k": p_at_k,
            "recall_at_k": r_at_k,
            "f1_at_k": f1_k,
            "num_relevant": sum(chunk_relevance),
            "total_relevant": total_rel,
            "num_retrieved": len(chunk_relevance),
            "chunk_scores": chunk_scores,
            "chunk_relevance": chunk_relevance,
        })

    mean_p = sum(precisions) / len(precisions)
    mean_r = sum(recalls) / len(recalls)
    mean_f1 = sum(f1s) / len(f1s)

    grid_results.append({
        "config": config_name,
        "k": k,
        "mean_precision_at_k": mean_p,
        "mean_recall_at_k": mean_r,
        "mean_f1_at_k": mean_f1,
        "std_precision_at_k": pd.Series(precisions).std(),
        "min_precision_at_k": min(precisions),
        "max_precision_at_k": max(precisions),
    })

    print(f"\n  ► {config_name} k={k}:  "
          f"Mean P@{k}={mean_p:.4f}  "
          f"Mean R@{k}={mean_r:.4f}  "
          f"Mean F1@{k}={mean_f1:.4f}")

print("\nGrid search complete!")


──────────────────────────────────────────────────────────────────────
Config: A_600_100 | k=3
──────────────────────────────────────────────────────────────────────
  Q01: P@3=1.0000  R@3=0.2500  F1@3=0.4000  [relevant: 3/3, total_rel: 12]
  Q02: P@3=0.3333  R@3=0.3333  F1@3=0.3333  [relevant: 1/3, total_rel: 3]
  Q03: P@3=0.3333  R@3=1.0000  F1@3=0.5000  [relevant: 1/3, total_rel: 1]
  Q04: P@3=0.3333  R@3=0.5000  F1@3=0.4000  [relevant: 1/3, total_rel: 2]
  Q05: P@3=0.0000  R@3=0.0000  F1@3=0.0000  [relevant: 0/3, total_rel: 3]
  Q06: P@3=0.3333  R@3=1.0000  F1@3=0.5000  [relevant: 1/3, total_rel: 1]
  Q07: P@3=0.3333  R@3=0.2500  F1@3=0.2857  [relevant: 1/3, total_rel: 4]
  Q08: P@3=0.0000  R@3=0.0000  F1@3=0.0000  [relevant: 0/3, total_rel: 2]
  Q09: P@3=1.0000  R@3=0.3333  F1@3=0.5000  [relevant: 3/3, total_rel: 9]
  Q10: P@3=1.0000  R@3=0.4286  F1@3=0.6000  [relevant: 3/3, total_rel: 7]
  Q11: P@3=0.3333  R@3=0.5000  F1@3=0.4000  [relevant: 1/3, total_rel: 2]
  Q12: P@3=0.6667 

In [37]:
df_grid = pd.DataFrame(grid_results)

print("=" * 70)
print("GRID SEARCH RESULTS — Mean Precision@K per (Config, k)")
print("=" * 70)
df_grid


GRID SEARCH RESULTS — Mean Precision@K per (Config, k)


,config,k,mean_precision_at_k,mean_recall_at_k,mean_f1_at_k,std_precision_at_k,min_precision_at_k,max_precision_at_k
0,A_600_100,3,0.466667,0.517460,0.410476,0.328537,0.0,1.000000
1,A_600_100,5,0.333333,0.575132,0.356069,0.269037,0.0,1.000000
2,A_600_100,7,0.295238,0.675397,0.357325,0.238231,0.0,0.857143
3,B_800_200,3,0.444444,0.544683,0.404911,0.325300,0.0,1.000000
4,B_800_200,5,0.360000,0.646429,0.389536,0.284856,0.0,1.000000
5,B_800_200,7,0.304762,0.713175,0.368992,0.246649,0.0,0.857143
6,C_900_150,3,0.444444,0.581349,0.438384,0.299912,0.0,1.000000
7,C_900_150,5,0.346667,0.696429,0.407509,0.255976,0.0,1.000000
8,C_900_150,7,0.276190,0.736508,0.357249,0.232032,0.0,0.857143
9,D_400_50,3,0.466667,0.463816,0.361811,0.328537,0.0,1.000000


In [27]:
print("=" * 70)
print("Pivot Tables: Mean Metrics per (Config, k)")
print("=" * 70)

print("\n── Precision@K ──")
pivot_p = df_grid.pivot(index="config", columns="k", values="mean_precision_at_k")
display(pivot_p)

print("\n── Recall@K ──")
pivot_r = df_grid.pivot(index="config", columns="k", values="mean_recall_at_k")
display(pivot_r)

print("\n── F1@K ──")
pivot_f1 = df_grid.pivot(index="config", columns="k", values="mean_f1_at_k")
display(pivot_f1)


Pivot Tables: Mean Metrics per (Config, k)

── Precision@K ──


k,3,5,7
config,,,
A_600_100,0.466667,0.333333,0.295238
B_800_200,0.444444,0.360000,0.304762
C_900_150,0.444444,0.346667,0.276190



── Recall@K ──


k,3,5,7
config,,,
A_600_100,0.517460,0.575132,0.675397
B_800_200,0.544683,0.646429,0.713175
C_900_150,0.581349,0.696429,0.736508



── F1@K ──


k,3,5,7
config,,,
A_600_100,0.410476,0.356069,0.357325
B_800_200,0.404911,0.389536,0.368992
C_900_150,0.438384,0.407509,0.357249


In [28]:
# best_idx = df_grid["mean_precision_at_k"].idxmax()
# best = df_grid.iloc[best_idx]

# print(f"Best Combination:")
# print(f"  Config:            {best['config']}")
# print(f"  k:                 {int(best['k'])}")
# print(f"  Mean Precision@K:  {best['mean_precision_at_k']:.4f}")
# print(f"  Std Precision@K:   {best['std_precision_at_k']:.4f}")
# print(f"  Min Precision@K:   {best['min_precision_at_k']:.4f}")
# print(f"  Max Precision@K:   {best['max_precision_at_k']:.4f}")


In [29]:
# df_grid.to_csv("grid_search_results.csv", index=False)

# print("Saved:")
# print("grid_search_results.csv")

In [38]:
!pip install rank_bm25 --break-system-packages -q

In [39]:
from rank_bm25 import BM25Okapi

def build_bm25_index(chunks):
    """chunks: list of langchain Document objects"""
    tokenized_corpus = [doc.page_content.lower().split() for doc in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    return bm25, chunks  # keep chunks aligned with tokenized_corpus by index

bm25_a, corpus_a = build_bm25_index(chunks_a)
bm25_b, corpus_b = build_bm25_index(chunks_b)
bm25_c, corpus_c = build_bm25_index(chunks_c)

BM25_INDEXES = {
    "A_600_100": (bm25_a, corpus_a),
    "B_800_200": (bm25_b, corpus_b),
    "C_900_150": (bm25_c, corpus_c),
}

In [40]:
# %%
def hybrid_search(query, config_name, k_final=5, pool_size=20, alpha=0.5):
    """
    alpha=1.0 -> pure dense
    alpha=0.0 -> pure BM25
    alpha=0.5 -> equal blend
    """
    vs = VECTORSTORES[config_name]
    bm25, corpus = BM25_INDEXES[config_name]

    # --- Dense side ---
    dense_results = retrieve_topk(query, k=pool_size, vs=vs)  # [(doc, distance), ...]
    # Chroma distance: smaller = more similar. Convert to a similarity-like score.
    dense_scores = {id(doc): 1 / (1 + dist) for doc, dist in dense_results}

    # --- BM25 side ---
    tokenized_query = query.lower().split()
    bm25_raw_scores = bm25.get_scores(tokenized_query)
    top_bm25_idx = sorted(range(len(bm25_raw_scores)), key=lambda i: bm25_raw_scores[i], reverse=True)[:pool_size]
    max_bm25 = max(bm25_raw_scores) if max(bm25_raw_scores) > 0 else 1
    bm25_results = [(corpus[i], bm25_raw_scores[i] / max_bm25) for i in top_bm25_idx]
    bm25_scores = {id(doc): score for doc, score in bm25_results}

    # --- Merge candidate pools from both methods ---
    all_docs = {id(doc): doc for doc, _ in dense_results}
    all_docs.update({id(doc): doc for doc, _ in bm25_results})

    merged = []
    for key, doc in all_docs.items():
        d_score = dense_scores.get(key, 0)
        b_score = bm25_scores.get(key, 0)
        combined = alpha * d_score + (1 - alpha) * b_score
        merged.append((doc, combined))

    merged.sort(key=lambda x: x[1], reverse=True)
    return merged[:k_final]

In [41]:
# %%
for qid in ["Q05", "Q08"]:
    q = next(x["question"] for x in TEST_QUESTIONS if x["id"] == qid)
    print(f"\n{'='*80}\n{qid}: {q}\n{'='*80}")

    print("\n--- Dense only (alpha=1.0) ---")
    for rank, (doc, score) in enumerate(hybrid_search(q, "A_600_100", k_final=5, alpha=1.0), 1):
        print(f"[{rank}] score={score:.4f} p.{doc.metadata.get('page_number')} :: {doc.page_content[:100]}...")

    print("\n--- Hybrid (alpha=0.5) ---")
    for rank, (doc, score) in enumerate(hybrid_search(q, "A_600_100", k_final=5, alpha=0.5), 1):
        print(f"[{rank}] score={score:.4f} p.{doc.metadata.get('page_number')} :: {doc.page_content[:100]}...")

    print("\n--- BM25 only (alpha=0.0) ---")
    for rank, (doc, score) in enumerate(hybrid_search(q, "A_600_100", k_final=5, alpha=0.0), 1):
        print(f"[{rank}] score={score:.4f} p.{doc.metadata.get('page_number')} :: {doc.page_content[:100]}...")


Q05: How does the NICE guideline define the "white-coat effect"?

--- Dense only (alpha=1.0) ---
[1] score=0.7118 p.17 :: 2.5 Certainty of evidence and strength of recommendations 
The GDG rated the certainty of evidence a...
[2] score=0.7069 p.17 :: (e.g. adverse effects). The strength of recommendations in this guideline was graded into two 
categ...
[3] score=0.6951 p.22 :: step to indicate treatment.
Evidence and rationale
The most direct evidence is derived from an indiv...
[4] score=0.6931 p.12 :: For a short explanation of why the committee deleted the recommendation on 
relaxation therapies and...
[5] score=0.6927 p.42 :: (13) 
Rubinstein A, Colantonio L, Bardach A, Caporale J, Martí SG, Kopitowski K, et al. Estimation o...

--- Hybrid (alpha=0.5) ---
[1] score=0.5000 p.50 :: Finding more information and committee 
details 
To find NICE guidance on related topics, including ...
[2] score=0.3904 p.19 :: antagonists: not for use in pregnancy, how to use for breastfeeding and 
cl

In [42]:
def search_corpus_for_phrase(phrase, chunks):
    matches = []
    for i, doc in enumerate(chunks):
        if phrase.lower() in doc.page_content.lower():
            matches.append((i, doc))
    return matches

# Check Q05
matches_q05 = search_corpus_for_phrase("white-coat", chunks_a)
print(f"'white-coat' found in {len(matches_q05)} chunks")
for i, doc in matches_q05:
    print(f"  chunk {i}, page {doc.metadata.get('page_number')}: {doc.page_content[:200]}")

# Check Q08
matches_q08 = search_corpus_for_phrase("potassium", chunks_a)
print(f"\n'potassium' found in {len(matches_q08)} chunks")
for i, doc in matches_q08:
    print(f"  chunk {i}, page {doc.metadata.get('page_number')}: {doc.page_content[:200]}")

'white-coat' found in 4 chunks
  chunk 102, page 15: • hypertension (with or without type 2 
diabetes) or 
• type 1 diabetes (regardless of 
albumin to creatinine ratio) 
Below 
150/90 
Recommendation 1.4.21 
NICE's guideline on type 1 
diabetes in adul
  chunk 113, page 26: daytime average or HBPM average blood pressure ranging from 135/85 mmHg to 149/
94 mmHg. 
Stage 2 hypertension 
Clinic blood pressure of 160/100 mmHg or higher but less than 180/120 mmHg and 
subseque
  chunk 119, page 32: arm blood pressure is associated with an increased risk of cardiovascular events, possibly 
due to vascular damage. 
ABPM correlates well with invasive blood pressure measurement and can identify both
  chunk 124, page 37: The committee decided to retain the 2011 recommendation on using clinic blood pressure, 
but also agreed that the updated guideline should support home monitoring for people 
who wish to use it. The c

'potassium' found in 7 chunks
  chunk 52, page 40: 6.2 Drug- and dose-specif